# Synthesis and Capture

We'll now examine how to generate and capture signals with the RF hardware integrated into the RFSoC. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time

from acadia.system import Acadia, StreamConfiguration
from acadia.channel import Channel
from acadia.arrays import ProceduralWaveform

No module named 'pyxrfdc'
No module named 'pyxrfclk'


We'll start by attaching to the board:

In [ ]:
acadia = Acadia()
acadia.attach()

Next, we'll set up the clocking system:

In [ ]:
acadia.configure_clocks(reference="internal")

We'll make sure that everything started up okay:

In [ ]:
acadia.get_clock_status()

In [ ]:
Channel.RFDC_status()

# System Configuration

In [2]:
acadia = Acadia()

pulse_channel = acadia.DAC(1)

def pulse_shape(out, t):
    out[:] = Channel.to_samples(0.99*np.ones(len(t), dtype=np.complex64))

pulse = ProceduralWaveform(pulse_channel, pulse_shape, pulse_channel)

capture_channel = acadia.ADC(1)
capture_data = ProceduralWaveform(capture_channel, None, acadia.PLDDR0Array)
capture_configuration = StreamConfiguration(capture_channel, module="bulk", acadia=acadia)

def configure():
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=2000e6)
    pulse_channel.set_vop(20000)
    
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    

In [3]:
# Create a sequence for the sequencer
def sequence(a):
    with a.synchronizer():
        a.generate(pulse_channel, pulse)
        a.capture(capture_configuration, capture_data)

pulse.allocate(1000e-9)
capture_data.allocate(5000e-9)

acadia.attach()
pulse.populate()
configure()

acadia.configure_stream(capture_configuration)

acadia.run(sequence)
time.sleep(0.1)
acadia.sequencer_halt()

In [ ]:
trace = Channel.from_samples(capture_data.memory())
times = capture_data.axis()*1e6

fig,ax = plt.subplots()
ax.plot(times, np.real(trace), label="Re")
ax.plot(times, np.imag(trace), label="Im")
ax.set_xlabel("Time (us)")
ax.set_ylabel("Amplitude (\%FS)")
ax.grid()
ax.legend()